# 05. Model Building

This notebook covers the fifth step in a typical QSAR workflow:
- Building QSAR models using various regression algorithms
- Training models on the training set
- Comparing different modeling approaches

ProQSAR supports multiple machine learning algorithms:
- Linear models: Multiple Linear Regression (MLR), Ridge, Lasso, Elastic Net
- Partial Least Squares (PLS)
- Tree-based models: Random Forest, Gradient Boosting, XGBoost, CatBoost
- Support Vector Machines (SVM)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from proqsar.Model.ModelDeveloper.model_developer import ModelDeveloper
from proqsar.Config.config import Config
import pickle
import os

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 5.1 Load Training Data

Load the training dataset from the previous step.

In [ ]:
# Load training data
train_path = '../Project/train_data.csv'
df_train = pd.read_csv(train_path)

print(f"Training data loaded: {df_train.shape}")
print(f"Number of compounds: {len(df_train)}")
print(f"Number of features: {df_train.shape[1] - 2}")
df_train.head()

## 5.2 Prepare Data for Modeling

Separate features and target variable.

In [ ]:
# Separate features and target
feature_cols = [col for col in df_train.columns if col not in ['Smiles', 'pChEMBL']]
X_train = df_train[feature_cols]
y_train = df_train['pChEMBL']
ids_train = df_train['Smiles']

print(f"Feature matrix shape: {X_train.shape}")
print(f"Target vector shape: {y_train.shape}")

## 5.3 Configure Model Developer

Configure the models to build and train.

In [ ]:
# Configure model developer
config = Config(
    model_developer={
        "models": ["rf", "xgb", "ridge"],  # RandomForest, XGBoost, Ridge
        # Other options: "lr" (LinearRegression), "lasso", "elasticnet", 
        # "svr", "pls", "gb" (GradientBoosting), "catboost"
        "task": "regression",
    }
)

print("Model developer configured!")
print(f"Models to train: {config.model_dev_config['models']}")
print(f"Task: {config.model_dev_config['task']}")

## 5.4 Initialize and Train Models

Initialize the ModelDeveloper and train multiple models.

In [ ]:
# Initialize model developer
model_developer = ModelDeveloper(
    activity_col="pChEMBL",
    id_col="Smiles",
    config=config,
    save_dir="../Project/ModelDeveloper",
    n_jobs=-1,  # Use all available cores
    random_state=42,
)

print("ModelDeveloper initialized!")

In [ ]:
# Train models with cross-validation
print("Training models...\n")
model_developer.fit(
    df_train,
    cv=5,  # 5-fold cross-validation
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']
)

print("\nModel training complete!")

## 5.5 Compare Model Performance

Compare the cross-validation performance of different models.

In [ ]:
# Get cross-validation results
cv_results = model_developer.report

if cv_results is not None:
    print("Cross-Validation Results:")
    print(cv_results)
    
    # Save CV results
    cv_results.to_csv('../Project/cv_results.csv', index=False)
    print("\nCV results saved to: ../Project/cv_results.csv")
else:
    print("No CV results available.")

In [ ]:
# Visualize model comparison
if cv_results is not None and 'model' in cv_results.columns:
    # Plot R² comparison
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # R² score
    if 'mean_test_r2' in cv_results.columns:
        r2_data = cv_results[['model', 'mean_test_r2']].copy()
        axes[0].bar(range(len(r2_data)), r2_data['mean_test_r2'])
        axes[0].set_xticks(range(len(r2_data)))
        axes[0].set_xticklabels(r2_data['model'], rotation=45)
        axes[0].set_ylabel('R² Score')
        axes[0].set_title('Model Performance: R² Score')
        axes[0].grid(True, alpha=0.3)
    
    # RMSE
    if 'mean_test_neg_mean_squared_error' in cv_results.columns:
        rmse_data = np.sqrt(-cv_results['mean_test_neg_mean_squared_error'])
        axes[1].bar(range(len(cv_results)), rmse_data)
        axes[1].set_xticks(range(len(cv_results)))
        axes[1].set_xticklabels(cv_results['model'], rotation=45)
        axes[1].set_ylabel('RMSE')
        axes[1].set_title('Model Performance: RMSE')
        axes[1].grid(True, alpha=0.3)
    
    # MAE
    if 'mean_test_neg_mean_absolute_error' in cv_results.columns:
        mae_data = -cv_results['mean_test_neg_mean_absolute_error']
        axes[2].bar(range(len(cv_results)), mae_data)
        axes[2].set_xticks(range(len(cv_results)))
        axes[2].set_xticklabels(cv_results['model'], rotation=45)
        axes[2].set_ylabel('MAE')
        axes[2].set_title('Model Performance: MAE')
        axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5.6 Select Best Model

Select the best performing model based on cross-validation results.

In [ ]:
# Get the best model
best_model = model_developer.model
best_model_name = model_developer.best_model_name if hasattr(model_developer, 'best_model_name') else "Unknown"

print(f"Best model: {best_model_name}")
print(f"Model type: {type(best_model).__name__}")

if cv_results is not None:
    best_idx = cv_results['mean_test_r2'].idxmax() if 'mean_test_r2' in cv_results.columns else 0
    print(f"\nBest model performance:")
    print(cv_results.iloc[best_idx])

## 5.7 Analyze Feature Importance (if available)

For tree-based models, analyze feature importance.

In [ ]:
# Check if model has feature_importances_ attribute
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("Top 20 Most Important Features:")
    print(feature_importance.head(20))
    
    # Plot feature importance
    plt.figure(figsize=(12, 8))
    plt.barh(range(20), feature_importance['importance'].head(20))
    plt.yticks(range(20), feature_importance['feature'].head(20))
    plt.xlabel('Feature Importance')
    plt.ylabel('Feature')
    plt.title(f'Top 20 Feature Importances ({best_model_name})')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # Save feature importance
    feature_importance.to_csv('../Project/feature_importance.csv', index=False)
    print("\nFeature importance saved to: ../Project/feature_importance.csv")
elif hasattr(best_model, 'coef_'):
    # For linear models
    coefficients = pd.DataFrame({
        'feature': feature_cols,
        'coefficient': np.abs(best_model.coef_)
    }).sort_values('coefficient', ascending=False)
    
    print("Top 20 Features by Absolute Coefficient:")
    print(coefficients.head(20))
    
    # Plot coefficients
    plt.figure(figsize=(12, 8))
    plt.barh(range(20), coefficients['coefficient'].head(20))
    plt.yticks(range(20), coefficients['feature'].head(20))
    plt.xlabel('|Coefficient|')
    plt.ylabel('Feature')
    plt.title(f'Top 20 Features by Absolute Coefficient ({best_model_name})')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # Save coefficients
    coefficients.to_csv('../Project/model_coefficients.csv', index=False)
    print("\nModel coefficients saved to: ../Project/model_coefficients.csv")
else:
    print("Feature importance not available for this model type.")

## 5.8 Save Trained Models

Save the trained models for later use.

In [ ]:
# Create models directory if it doesn't exist
os.makedirs('../Project/models', exist_ok=True)

# Save the best model
best_model_path = '../Project/models/best_model.pkl'
with open(best_model_path, 'wb') as f:
    pickle.dump(best_model, f)

print(f"Best model saved to: {best_model_path}")

# Save all trained models
if hasattr(model_developer, 'models_'):
    for model_name, model in model_developer.models_.items():
        model_path = f'../Project/models/{model_name}_model.pkl'
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)
        print(f"{model_name} model saved to: {model_path}")

## 5.9 Summary

Summarize the model building process.

In [ ]:
print("="*60)
print("MODEL BUILDING SUMMARY")
print("="*60)
print(f"Training set size: {len(df_train)}")
print(f"Number of features: {len(feature_cols)}")
print(f"\nModels trained: {config.model_dev_config['models']}")
print(f"Best model: {best_model_name}")

if cv_results is not None and 'mean_test_r2' in cv_results.columns:
    best_idx = cv_results['mean_test_r2'].idxmax()
    print(f"\nBest model CV performance:")
    if 'mean_test_r2' in cv_results.columns:
        print(f"  R²: {cv_results.iloc[best_idx]['mean_test_r2']:.4f}")
    if 'mean_test_neg_mean_squared_error' in cv_results.columns:
        rmse = np.sqrt(-cv_results.iloc[best_idx]['mean_test_neg_mean_squared_error'])
        print(f"  RMSE: {rmse:.4f}")
    if 'mean_test_neg_mean_absolute_error' in cv_results.columns:
        mae = -cv_results.iloc[best_idx]['mean_test_neg_mean_absolute_error']
        print(f"  MAE: {mae:.4f}")

print(f"\nModels saved to: ../Project/models/")
print("\nModels are ready for validation!")
print("="*60)

## Next Steps

The QSAR models have been built and trained. The next notebook (06_model_validation.ipynb) will:
- Validate models using internal cross-validation
- Evaluate models on the external test set
- Analyze performance metrics and prediction quality